# Project FORESIGHT — 02 Seasonal-Naive Baseline

This notebook implements the Seasonal-Naive baseline model, computes its forecasting performance (WAPE and Bias) on the validation set, and records it as our benchmark.

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Load Cleaned Dataset

In [ ]:
df = pd.read_csv("../data/processed/clean_sales.csv")
df["date_dt"] = pd.to_datetime(df["date"])
print(f"Data loaded: {len(df)} rows across {df['sku_id'].nunique()} SKUs")

## 2. Implement Seasonal-Naive Forecast

For our baseline, the demand forecast for a SKU on day $t$ is the average demand of the same day-of-week over the past 4 weeks (28 days).

In [ ]:
def calculate_wape(actual, forecast):
    return np.sum(np.abs(actual - forecast)) / np.sum(actual)

def calculate_bias(actual, forecast):
    return np.sum(forecast - actual) / np.sum(actual)

# Let's split a test set of the last 6 weeks (42 days) of history
end_date = df["date_dt"].max()
split_date = end_date - timedelta(days=42)

train_df = df[df["date_dt"] < split_date]
test_df = df[df["date_dt"] >= split_date].copy()

print(f"Train period: {train_df['date'].min()} to {train_df['date'].max()}")
print(f"Test period: {test_df['date'].min()} to {test_df['date'].max()}")

## 3. Compute Baseline Predictions

In [ ]:
# Pre-calculate baseline values from the last 28 days of training data
max_train_date = train_df["date_dt"].max()
recent_train = train_df[train_df["date_dt"] > (max_train_date - timedelta(days=28))]

baseline_lookup = recent_train.groupby(["sku_id", "day_of_week"])["units_sold"].mean().to_dict()
sku_lookup = recent_train.groupby("sku_id")["units_sold"].mean().to_dict()
global_mean = recent_train["units_sold"].mean()

test_df["baseline_pred"] = test_df.apply(
    lambda r: baseline_lookup.get((r["sku_id"], r["day_of_week"]), sku_lookup.get(r["sku_id"], global_mean)),
    axis=1
)
test_df["baseline_pred"] = test_df["baseline_pred"].clip(lower=0.0)

wape = calculate_wape(test_df["units_sold"].values, test_df["baseline_pred"].values)
bias = calculate_bias(test_df["units_sold"].values, test_df["baseline_pred"].values)

print(f"Seasonal-Naive Baseline Validation Metrics (Last 6 weeks):")
print(f"WAPE: {wape:.4f}")
print(f"Bias: {bias:.4f}")

## 4. Visualizing Baseline Performance
Let's visualize actual vs baseline forecast for a sample SKU.

In [ ]:
sample_sku = test_df["sku_id"].iloc[0]
sku_test = test_df[test_df["sku_id"] == sample_sku].sort_values("date")

plt.figure(figsize=(12, 5))
plt.plot(sku_test["date"], sku_test["units_sold"], label="Actual Sales", marker="o")
plt.plot(sku_test["date"], sku_test["baseline_pred"], label="Baseline Forecast", linestyle="--", marker="x")
plt.title(f"Seasonal-Naive Baseline vs Actuals for SKU: {sample_sku}")
plt.xlabel("Date")
plt.ylabel("Units")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()